## 1 Setup and data

Run the installation cell once in a fresh Python 3 environment. Locally, open the notebook from `notebooks/`, beside `data/`.

**Colab upload:** use the Files sidebar to upload the three `banking77_m04_*.csv` files. `DATA_DIR` -> Path to our course data folder.

In [ ]:
%pip -q install "torch>=2.6,<3" "transformers==4.57.6" "pandas>=2.2,<4" "scikit-learn>=1.5,<2" "matplotlib>=3.9,<4" sentencepiece

In [ ]:
from pathlib import Path  # handles file paths

import matplotlib.pyplot as plt  # makes plots
import pandas as pd  # works with tables
import torch  # runs PyTorch models
from IPython.display import display  # shows output nicely
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, f1_score  # evaluation metrics
from transformers import pipeline  # loads Hugging Face pipelines

DEVICE = 0 if torch.cuda.is_available() else -1  # use GPU if available
BATCH_SIZE = 8  # examples per batch

torch.manual_seed(42)  # fixes randomness
torch.set_num_threads(4)  # limits CPU threads
pd.set_option("display.max_colwidth", 140)  # widens text display
print("Device:", torch.cuda.get_device_name(0) if DEVICE == 0 else "CPU")  # prints device

### Dataset: 488 messages from BANKING77

The activity retains four BANKING77 intents and three separate roles for labeled data.

| File | Messages | Role in this activity |
|---|---:|---|
| Examples | 8 · 2/class | Included in the few-shot prompt; never used in the zero-shot prompt |
| Development | 80 · 20/class | Practice comparison and error inspection before final evaluation |
| Evaluation | 400 · 100/class | Held-out comparison after prompts are fixed |

| Classroom label | Source intent | Meaning |
|---|---|---|
| `delivery` | `card_arrival` | A card has not arrived or the customer asks when it will arrive |
| `lost_card` | `lost_or_stolen_card` | An existing card was lost or stolen |
| `pending_payment` | `pending_card_payment` | A card payment is pending or incomplete |
| `cash_fee` | `cash_withdrawal_charge` | A fee was charged for withdrawing cash |

Examples and development use distinct source-training rows. Evaluation combines all 160 source-test messages for these intents with 240 additional source-training messages excluded from examples and development. This is a **custom classroom holdout**, not the official BANKING77 test split. The sample is balanced by design and does not represent all banking requests.

**Attribution:** [BANKING77, PolyAI / Casanueva et al. (2020)](https://github.com/PolyAI-LDN/task-specific-datasets), *Efficient Intent Detection with Dual Sentence Encoders*, **CC BY 4.0**.


In [ ]:
DATA_DIR = Path("/content/drive/MyDrive/a-ncsu-courses/DSA495-2026/Data/M04_Classification02")  # Colab upload: Path("/content")

development = pd.read_csv(DATA_DIR / "banking77_m04_development_72.csv")
evaluation = pd.read_csv(DATA_DIR / "banking77_m04_evaluation.csv")

In [ ]:
development = development.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffles development
evaluation = evaluation.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffles evaluation

In [ ]:
LABELS = ["delivery", "lost_card", "pending_payment", "cash_fee"]  # target classes

counts = pd.DataFrame({  # builds a table
    "development": development["label"].value_counts(),  # counts development labels
    "evaluation": evaluation["label"].value_counts(),  # counts evaluation labels
}).reindex(LABELS)  # orders rows by labels
display(counts)  # shows the table

## 2 Zero-shot classification

Start with **instructions, category definitions, and one message to classify**. No labeled example messages are supplied.

We use [Flan-T5-base](https://huggingface.co/google/flan-t5-base), an instruction-tuned encoder–decoder model (Apache 2.0). Its weights stay fixed. The four allowed answer codes are A = delivery, B = lost card, C = pending payment, and D = cash fee. Category definitions are fixed throughout the activity.

The instruction to return a letter guides generation. We allow one generated token; A/B/C/D are each single tokens for this tokenizer. The code retains raw outputs and marks any unmapped response as `INVALID`.


In [ ]:
MODEL_ID = "google/flan-t5-base"  # model name
MODEL_REVISION = "c782cba52f8ea6a704240578055cf1c3fc2f2ca9"  # fixed model version

classifier = pipeline(  # creates pipeline
    "text2text-generation",  # text-to-text task
    model=MODEL_ID,  # uses chosen model
    revision=MODEL_REVISION,  # uses fixed version
    device=DEVICE,  # uses GPU or CPU
)

In [ ]:
CODE_TO_LABEL = {"A": "delivery", "B": "lost_card", "C": "pending_payment", "D": "cash_fee"}  # maps codes to labels

INSTRUCTIONS = (  # classification prompt
    "Classify the customer message. Answer with one letter: A, B, C, or D.\n"  # task instruction
    "A: A bank card has not arrived, or the customer asks when it will arrive.\n"  # delivery meaning
    "B: The customer's bank card was lost or stolen.\n"  # lost card meaning
    "C: A card payment is pending or has not completed.\n"  # pending payment meaning
    "D: An extra fee was charged for withdrawing cash.\n\n"  # cash fee meaning
)

def make_zero_prompt(text):  # builds zero-shot prompt
    return INSTRUCTIONS + f"Message: {text}\nAnswer:"  # adds message and answer cue

example_text = development.loc[0, "text"]  # gets sample text
print(make_zero_prompt(example_text))  # prints prompt

In [ ]:
# Predict one message and inspect the raw answer before running the full set.
single_output = classifier(  # runs the model on one prompt
    make_zero_prompt(example_text),  # creates the full classification prompt
    max_new_tokens=1,  # limits the answer to one generated token
    do_sample=False  # uses deterministic output instead of random sampling
)
print(single_output)
print("Reference label:", development.loc[0, "label"])


**Your turn:** Why is this zero-shot even though the prompt lists four categories?


In [ ]:
zero_prompts = [make_zero_prompt(text) for text in development["text"]]  # makes prompts for all development texts

In [ ]:
zero_outputs = classifier(zero_prompts, max_new_tokens=1, do_sample=False, batch_size=BATCH_SIZE)  # runs model in batches

In [ ]:
development["zero_raw"] = [row["generated_text"] for row in zero_outputs]  # stores raw model outputs

In [ ]:
development["zero_raw"] = [row["generated_text"] for row in zero_outputs]  # stores raw model outputs
development["zero_shot"] = development["zero_raw"].str.strip().str.upper().map(CODE_TO_LABEL).fillna("INVALID")  # converts letters to labels
development[["text", "label", "zero_raw", "zero_shot"]].head(8)  # previews first 8 rows

In [ ]:
development["zero_shot"] = (  # creates cleaned zero-shot predictions
    development["zero_raw"]  # starts with raw model output
    .str.strip()  # removes extra spaces/newlines (just in case there's any)
    .str.upper()  # makes letters uppercase (just in case there's any)
    .map(CODE_TO_LABEL)  # converts A/B/C/D to label names
    .fillna("INVALID")  # marks unmapped outputs as invalid
)

In [ ]:
development[["text", "label", "zero_raw", "zero_shot"]] #

##### Inspect the Performance on the Development Set Before Moving On -- See what you may like to do to enhance the performance

In [ ]:
zero_accuracy = accuracy_score(development["label"], development["zero_shot"])
zero_f1 = f1_score(development["label"], development["zero_shot"], labels=LABELS, average="macro", zero_division=0)
print("Zero-shot development accuracy:", round(zero_accuracy, 3))
print("Zero-shot development macro-F1:", round(zero_f1, 3))
print(classification_report(development["label"], development["zero_shot"], labels=LABELS, zero_division=0))

In [ ]:
zero_errors = development[development["label"] != development["zero_shot"]]
print("Zero-shot development errors:", len(zero_errors))
zero_errors[["text", "label", "zero_shot"]].head(8)

**Pause before moving on:** Explain the complete zero-shot workflow: construct a prompt, generate an answer, map it to a label, and compare with the reference.


## 3 Few-shot classification

Now add labeled message–answer demonstrations before the message being classified. The model and instructions are the same as in Section 2.

We will increase **examples per class**, maintaining representation of all four classes:

| Setting | Examples per class | Total demonstrations in each prompt |
|---|---:|---:|
| `few_1_per_class` | 1 | 4 (four-shot) |
| `few_2_per_class` | 2 | 8 (eight-shot) |
| `few_4_per_class` | 4 | 16 (sixteen-shot) |

Each larger set contains every example from the smaller set in the same relative order. This is a single fixed sequence of examples; differences may reflect the content of the added messages as well as their number. Model weights are never updated.


In [ ]:
example_pool = pd.read_csv(DATA_DIR / "banking77_m04_examples_16.csv")

In [ ]:
example_pool = example_pool.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffles examples

examples_1 = example_pool[example_pool["example_rank"] <= 1]
examples_2 = example_pool[example_pool["example_rank"] <= 2]
examples_4 = example_pool[example_pool["example_rank"] <= 4]

In [ ]:
display(pd.DataFrame({
    "1_per_class": examples_1["label"].value_counts(),
    "2_per_class": examples_2["label"].value_counts(),
    "4_per_class": examples_4["label"].value_counts(),
}).reindex(LABELS))

In [ ]:
LABEL_TO_CODE = {label: code for code, label in CODE_TO_LABEL.items()}  # reverses label map

def build_demonstrations(rows):  # builds few-shot examples
    demonstrations = ""  # starts empty text
    for row in rows.itertuples():  # loops over rows
        demonstrations += f"Message: {row.text}\nAnswer: {LABEL_TO_CODE[row.label]}\n\n"  # adds example answer
    return demonstrations  # returns examples text

demo_1 = build_demonstrations(examples_1)  # builds 1-shot demo text
demo_2 = build_demonstrations(examples_2)  # builds 2-shot demo text
demo_4 = build_demonstrations(examples_4)  # builds 4-shot demo text

In [ ]:
def make_few_prompt(text, demonstrations):  # builds few-shot prompt
    return INSTRUCTIONS + demonstrations + f"Message: {text}\nAnswer:"  # adds demos and new message

In [ ]:
print(make_few_prompt(example_text, demo_1))  # prints sample few-shot prompt

### 3.1 · 1 example per class (4 total)

Classify the same 72 development messages with this fixed example set.


In [ ]:
few_1_prompts = [make_few_prompt(text, demo_1) for text in development["text"]]  # makes 1-per-class prompts
few_1_outputs = classifier(few_1_prompts, max_new_tokens=1, do_sample=False, batch_size=BATCH_SIZE)  # runs model in batches

In [ ]:
development["few_1_raw"] = [row["generated_text"] for row in few_1_outputs]  # stores raw outputs
development["few_1_per_class"] = development["few_1_raw"].str.strip().str.upper().map(CODE_TO_LABEL).fillna("INVALID")  # converts codes to labels

In [ ]:
print("Accuracy:", round(accuracy_score(development["label"], development["few_1_per_class"]), 3))  # prints accuracy
print("Macro-F1:", round(f1_score(development["label"], development["few_1_per_class"], labels=LABELS, average="macro", zero_division=0), 3))  # prints macro-F1


In [ ]:
development[["text", "label", "few_1_raw", "few_1_per_class"]]

### 3.2 · 2 examples per class (8 total)

Classify the same 72 development messages with this fixed example set.


In [ ]:
few_2_prompts = [make_few_prompt(text, demo_2) for text in development["text"]]  # makes 2-per-class prompts
few_2_outputs = classifier(few_2_prompts, max_new_tokens=1, do_sample=False, batch_size=BATCH_SIZE)  # runs model in batches
development["few_2_raw"] = [row["generated_text"] for row in few_2_outputs]  # stores raw outputs
development["few_2_per_class"] = development["few_2_raw"].str.strip().str.upper().map(CODE_TO_LABEL).fillna("INVALID")  # converts codes to labels
print("Accuracy:", round(accuracy_score(development["label"], development["few_2_per_class"]), 3))  # prints accuracy
print("Macro-F1:", round(f1_score(development["label"], development["few_2_per_class"], labels=LABELS, average="macro", zero_division=0), 3))  # prints macro-F1
development[["text", "label", "few_2_raw", "few_2_per_class"]]# previews first 4 rows

### 3.3 · 4 examples per class (16 total)

Classify the same 72 development messages with this fixed example set.


In [ ]:
few_4_prompts = [make_few_prompt(text, demo_4) for text in development["text"]]  # makes 4-per-class prompts
few_4_outputs = classifier(few_4_prompts, max_new_tokens=1, do_sample=False, batch_size=BATCH_SIZE)  # runs model in batches
development["few_4_raw"] = [row["generated_text"] for row in few_4_outputs]  # stores raw outputs
development["few_4_per_class"] = development["few_4_raw"].str.strip().str.upper().map(CODE_TO_LABEL).fillna("INVALID")  # converts codes to labels
print("Accuracy:", round(accuracy_score(development["label"], development["few_4_per_class"]), 3))  # prints accuracy
print("Macro-F1:", round(f1_score(development["label"], development["few_4_per_class"], labels=LABELS, average="macro", zero_division=0), 3))  # prints macro-F1
development[["text", "label", "few_4_raw", "few_4_per_class"]]

### 3.4 · Compare development results

After running each workflow separately, bring its predictions together. All rows below are scored against the same reference messages.


In [ ]:
METHODS = ["zero_shot", "few_1_per_class", "few_2_per_class", "few_4_per_class"]  # prediction columns
development_scores = pd.DataFrame(index=METHODS)  # creates score table

In [ ]:
development_scores

In [ ]:
for method in METHODS:  # loops through methods
    development_scores.loc[method, "accuracy"] = accuracy_score(development["label"], development[method])  # saves accuracy
    development_scores.loc[method, "macro_f1"] = f1_score(development["label"], development[method], labels=LABELS, average="macro", zero_division=0)  # saves macro-F1
    development_scores.loc[method, "valid_rate"] = development[method].isin(LABELS).mean()  # saves percent valid

development_scores.round(3)  # shows rounded scores

In [ ]:
development_changes = development[development[METHODS].nunique(axis=1) > 1]
print("Messages with different predictions:", len(development_changes))
development_changes[["text", "label"] + METHODS].groupby("label", sort=False).head(2)


**Your turn:** Does performance improve steadily as examples increase? Read a changed prediction and explain the evidence.


## 4 Final evaluation on 400 messages

Run the four fixed settings on the same evaluation messages. The prompts, example membership/order, model revision, and decoding settings are now fixed. This set has an earlier evaluation history; report it as classroom evidence, not a new independent benchmark. Further prompt changes belong on development data or require a fresh evaluation set.


In [ ]:
evaluation_zero_prompts = [make_zero_prompt(text) for text in evaluation["text"]]  # makes zero-shot prompts
evaluation_zero_outputs = classifier(evaluation_zero_prompts, max_new_tokens=1, do_sample=False, batch_size=BATCH_SIZE)  # runs model in batches
evaluation["zero_raw"] = [row["generated_text"] for row in evaluation_zero_outputs]  # stores raw outputs
evaluation["zero_shot"] = evaluation["zero_raw"].str.strip().str.upper().map(CODE_TO_LABEL).fillna("INVALID")  # converts codes to labels

In [ ]:
# 1 example per class
few_1_prompts = [make_few_prompt(text, demo_1) for text in evaluation["text"]]
few_1_outputs = classifier(few_1_prompts, max_new_tokens=1, do_sample=False, batch_size=BATCH_SIZE)
evaluation["few_1_raw"] = [row["generated_text"] for row in few_1_outputs]
evaluation["few_1_per_class"] = evaluation["few_1_raw"].str.strip().str.upper().map(CODE_TO_LABEL).fillna("INVALID")

# 2 examples per class
few_2_prompts = [make_few_prompt(text, demo_2) for text in evaluation["text"]]
few_2_outputs = classifier(few_2_prompts, max_new_tokens=1, do_sample=False, batch_size=BATCH_SIZE)
evaluation["few_2_raw"] = [row["generated_text"] for row in few_2_outputs]
evaluation["few_2_per_class"] = evaluation["few_2_raw"].str.strip().str.upper().map(CODE_TO_LABEL).fillna("INVALID")

# 4 examples per class
few_4_prompts = [make_few_prompt(text, demo_4) for text in evaluation["text"]]
few_4_outputs = classifier(few_4_prompts, max_new_tokens=1, do_sample=False, batch_size=BATCH_SIZE)
evaluation["few_4_raw"] = [row["generated_text"] for row in few_4_outputs]
evaluation["few_4_per_class"] = evaluation["few_4_raw"].str.strip().str.upper().map(CODE_TO_LABEL).fillna("INVALID")

### 4.1 Evaluation Results

In [ ]:
evaluation_zero_results = evaluation[["text", "label", "zero_raw", "zero_shot"]].copy()  # zero-shot results
evaluation_few_1_results = evaluation[["text", "label", "few_1_raw", "few_1_per_class"]].copy()  # 1 example per class results
evaluation_few_2_results = evaluation[["text", "label", "few_2_raw", "few_2_per_class"]].copy()  # 2 examples per class results
evaluation_few_4_results = evaluation[["text", "label", "few_4_raw", "few_4_per_class"]].copy()  # 4 examples per class results

In [ ]:
evaluation_scores = pd.DataFrame(index=METHODS)  # creates score table

for method in METHODS:  # loops through methods
    evaluation_scores.loc[method, "accuracy"] = accuracy_score(evaluation["label"], evaluation[method])  # saves accuracy
    evaluation_scores.loc[method, "macro_f1"] = f1_score(evaluation["label"], evaluation[method], labels=LABELS, average="macro", zero_division=0)  # saves macro-F1
    evaluation_scores.loc[method, "valid_rate"] = evaluation[method].isin(LABELS).mean()  # saves percent valid

display(evaluation_scores.round(3))  # shows rounded scores

In [ ]:
print("zero_shot")
print(classification_report(evaluation["label"], evaluation["zero_shot"], labels=LABELS, digits=3, zero_division=0))

print("few_1_per_class")
print(classification_report(evaluation["label"], evaluation["few_1_per_class"], labels=LABELS, digits=3, zero_division=0))

print("few_2_per_class")
print(classification_report(evaluation["label"], evaluation["few_2_per_class"], labels=LABELS, digits=3, zero_division=0))

print("few_4_per_class")
print(classification_report(evaluation["label"], evaluation["few_4_per_class"], labels=LABELS, digits=3, zero_division=0))

### 4.1 Create the Confusion Matrix

In [ ]:
has_invalid = (evaluation[METHODS] == "INVALID").any().any()  # checks if any method predicted INVALID

In [ ]:
matrix_labels = LABELS + ["INVALID"] if has_invalid else LABELS  # includes INVALID only if needed
display_labels = ["Delivery", "Lost card", "Pending", "Cash fee", "Invalid"] if has_invalid else ["Delivery", "Lost card", "Pending", "Cash fee"]  # matches plot labels

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 10))  # creates 2x2 plot grid

method_titles = {  # nicer subplot titles
    "zero_shot": "Zero-shot",
    "few_1_per_class": "1 example per class",
    "few_2_per_class": "2 examples per class",
    "few_4_per_class": "4 examples per class",
}

for method, ax in zip(METHODS, axes.flat):  # loops through methods and axes
    ConfusionMatrixDisplay.from_predictions(  # makes confusion matrix
        evaluation["label"], evaluation[method],  # uses true and predicted labels
        labels=matrix_labels,  # sets label order
        display_labels=display_labels,  # labels shown on plot
        cmap="Blues",  # uses blue colors
        colorbar=False,  # hides colorbar
        values_format="d",  # shows whole numbers
        ax=ax,  # draws on current subplot
    )
    ax.set_title(method_titles[method])  # uses standardized titles instead of `(method)`
    ax.tick_params(axis="x", rotation=30)  # rotates x labels

fig.suptitle("400 messages per setting · rows = reference, columns = prediction")  # adds main title
fig.tight_layout()  # adjusts spacing
plt.show()  # displays plot

**Your turn:** Use one confusion matrix above to calculate precision and recall by hand.

For the **2 examples per class** setting and answer:

1. What is the recall for **Delivery**?
2. What is the precision for **Delivery**?
3. What's the F1 for **Delivery**?

[optional]

4. Pick one other class and calculate its precision and recall.
5. Which number is easier to calculate from this table: precision or recall? Why?

### 4.2 Changed Predictions Across Methods

In [ ]:
evaluation_changes = evaluation[evaluation[METHODS].nunique(axis=1) > 1]  # keeps rows where methods disagree
print("Evaluation messages with different predictions:", len(evaluation_changes))  # prints number of disagreements
evaluation_changes[["text", "label"] + METHODS].groupby("label", sort=False).head(2)  # shows up to 2 examples per label

### 4.3 Messages misclassified by at least one setting

In [ ]:
errors = evaluation[evaluation[METHODS].ne(evaluation["label"], axis=0).any(axis=1)]  # keeps rows missed by at least one method
print("Messages misclassified by at least one setting:", len(errors))  # prints number of rows with any error
errors[["text", "label"] + METHODS].groupby("label", sort=False).head(2)  # shows up to 2 error examples per label

**Your turn: evidence-based recommendation**

1. Compare accuracy and macro-F1 for zero-shot and all three few-shot settings.
2. Use a class report or confusion matrix to identify one weakness.
3. Read two disagreements and cite words that support the reference or predicted labels.
4. Explain whether adding more examples helped consistently.
5. State one limitation and propose a follow-up using development data or a fresh evaluation set.


## 5. Save Results

In [ ]:
OUTPUT_DIR = Path("Your Google Drive Directory")
OUTPUT_DIR.mkdir(exist_ok=True)

evaluation.to_csv(OUTPUT_DIR / "evaluation_predictions.csv", index=False)
evaluation_scores.to_csv(OUTPUT_DIR / "evaluation_metrics.csv", index_label="method")

In [ ]:
# Your Turn: Save the Evaluation Performance Results tables as well.

**Exit ticket:** In 3–5 sentences, define zero-shot and few-shot prompting, distinguish examples per class from total shots, state what stayed fixed, and support your conclusion with a metric and one message.

> **Optional NLI connection:** [Natural language inference (NLI)](https://huggingface.co/tasks/zero-shot-classification) classifiers score a message against category hypotheses. That is another zero-shot mechanism. This lesson uses a single generative model so students can add examples to its input and evaluate the change. -- e.g., facebook/bart-large-mnli

**Sources:** [Flan-T5-base model card](https://huggingface.co/google/flan-t5-base); [Chapter 4 companion notebook](https://github.com/handsOnLLM/Hands-On-Large-Language-Models/blob/main/chapter04/Chapter%204%20-%20Text%20Classification.ipynb). Code is independently written for this activity. See `DSA495-M04-preparation-notes.md` for validation and `../data/README.md` for source attribution and reproducible sampling.
